# HKEX tools
Two tools in this one notebook. **Click a tool's code cell, then Run** (▶ or Shift+Enter). Each cell is complete on its own and pulls fresh data from HKEX every time; nothing else needs to be run first.

| | What it does | You get |
|---|---|---|
| **Tool 1** · Rights & options email | HKEX's latest *Dividends & Other Entitlements* list → every stock with an entry mentioning **Right** or **Option**, with all of that stock's rows | an Outlook draft (never sent), shown below the cell too |
| **Tool 2** · Dividend currency elections | HKEX filings of the last five years → every company that let holders choose the dividend currency, **HKD & RMB** first | a table below the cell, plus CSV and Markdown copies in `hkex_out` |

Python 3.8+. Tool 1 uses Outlook through pywin32 on Windows; tool 2 installs `pypdf` by itself the first time.

## Tool 1 · Rights & options email
- Source: [HKEX Dividends & Other Entitlements](https://www3.hkexnews.hk/reports/doe/eent.htm). HKEX rebuilds it at irregular times (09:30-17:15 HK time, not every business day); the run prints HKEX's own **Updated** time.
- Keeps every stock whose Description contains RIGHT or OPTION (any case) **with all its rows** (e.g. both CITIC BANK rows), in HKEX's order and columns; `*` = HKEX's own flag.
- Windows + Outlook: the draft is saved in Drafts and opened. Anywhere else: an `.eml` draft in `hkex_out`.
- Put recipients in `TO` / `CC` at the top of the cell if you want the draft addressed.

In [ ]:
# TOOL 1 - HKEX rights & options email.  Edit CONFIG if you like, then Run this cell.
import datetime as dt
import email.utils
import gzip
import html
import http.client
import itertools
import os
import re
import ssl
import subprocess
import sys
import tempfile
import time
import urllib.request
from email.message import EmailMessage
from html.parser import HTMLParser

# ================================================================ CONFIG ===
URL = 'https://www3.hkexnews.hk/reports/doe/eent.htm'
KEYWORDS = ('RIGHT', 'OPTION')    # kept if the Description contains any of these, any case
TO = ''                           # 'a@nomura.com; b@nomura.com' - or leave blank, fill in Outlook
CC = ''
SUBJECT = 'HKEX Entitlements – Rights & Options – {date}'
GREETING = 'Hi all,'
OPEN_DRAFT = True                 # open the draft for review; False = only save it to Drafts
try:
    HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:                 # pasted straight into a notebook cell
    HERE = os.getcwd()
OUT_DIR = os.path.join(HERE, 'hkex_out')   # the .eml lands here when Outlook is not reachable

# ================================================================ ENGINE ===
HKT = dt.timezone(dt.timedelta(hours=8), 'HKT')        # Hong Kong has no daylight saving
UA = ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
      '(KHTML, like Gecko) Chrome/124.0 Safari/537.36')
NO_CACHE = {'Cache-Control': 'no-cache', 'Pragma': 'no-cache'}
DAYS = ('Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun')      # not %a / %b: those follow the
MONTHS = ('Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',            # PC's language setting
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec')
FONT = 'font-family:Arial,Helvetica,sans-serif;'
INK, MUTED, RULE, HEAD, BAND, FLAG = '#1f2937', '#6b7280', '#dfe4ea', '#1f3a5f', '#f3f6fa', '#c0392b'


class Stop(Exception):
    """A failure one sentence explains - printed instead of a traceback."""


# ----------------------------------------------------------------- fetch ---
def fetch(url=URL):
    """(page, updated, note) - always a fresh copy: a one-off query string plus
    no-cache headers get past every cache on the way (office proxy, HKEX's CDN,
    Windows).  `updated` is HKEX's own Last-Modified time for the page, in HKT."""
    fresh = f'{url}{"&" if "?" in url else "?"}_={int(time.time())}'
    try:
        page, updated, note = _python_get(fresh)
    except (OSError, http.client.HTTPException) as e:
        reason = getattr(e, 'reason', e)
        if sys.platform != 'win32':
            raise Stop(f'could not reach HKEX ({reason}) - check the network or proxy, then run again')
        try:    # e.g. an office proxy that wants a sign-in: go the way the browser does
            page, updated = _windows_get(fresh)
            note = 'fetched through Windows networking - Python could not get out directly'
        except Exception as e2:
            raise Stop(f'could not reach HKEX (Python: {reason}; Windows: {e2}) - '
                       f'check the network, then run again')
    return page, _hkt(updated), note


def _python_get(url):
    """Certificates: the system's, then certifi's, then none - this is a public
    read-only page, better fetched unchecked than not at all."""
    contexts = [('', ssl.create_default_context())]
    try:
        import certifi
        contexts.append(('', ssl.create_default_context(cafile=certifi.where())))
    except ImportError:
        pass
    contexts.append(('certificate check skipped - this Python has no CA bundle',
                     ssl._create_unverified_context()))
    request = urllib.request.Request(url, headers=dict(NO_CACHE, **{'User-Agent': UA}))
    for note, context in contexts:
        try:
            with urllib.request.urlopen(request, timeout=20, context=context) as response:
                raw, updated = response.read(), response.headers.get('Last-Modified')
            break
        except OSError as e:
            if not isinstance(getattr(e, 'reason', e), ssl.SSLCertVerificationError):
                raise
            refused = e
    else:
        raise refused                          # every certificate route refused
    if raw[:2] == b'\x1f\x8b':              # gzipped although we did not ask (some proxies do)
        raw = gzip.decompress(raw)
    try:
        return raw.decode('utf-8'), updated, note
    except UnicodeDecodeError:
        return raw.decode('cp1252', errors='replace'), updated, note


def _windows_get(url):
    """The page through Windows' own web stack (MSXML/WinINet): the browser's proxy
    settings and sign-in.  Raises if that fails too."""
    import win32com.client
    xhr = win32com.client.Dispatch('MSXML2.XMLHTTP.6.0')
    xhr.open('GET', url, False)
    for name, value in NO_CACHE.items():
        xhr.setRequestHeader(name, value)
    xhr.send()
    if xhr.status != 200:
        raise OSError(f'HTTP {xhr.status}')
    try:
        updated = xhr.getResponseHeader('Last-Modified')
    except Exception:
        updated = None
    return xhr.responseText, updated


def _hkt(http_date):
    """An HTTP date header as an HKT datetime; None if it is missing or odd."""
    try:
        return email.utils.parsedate_to_datetime(http_date).astimezone(HKT)
    except (TypeError, ValueError):
        return None


# ----------------------------------------------------------------- parse ---
class _Cells(HTMLParser):
    """Every <tr> as a list of cell texts, <br> kept as a newline.  HKEX leaves
    cells unclosed, so a new <td> or <tr> closes whatever is still open."""

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.rows, self._row, self._cell = [], None, None

    def handle_starttag(self, tag, attrs):
        if tag == 'tr':
            self._end_row()
            self._row = []
        elif tag in ('td', 'th'):
            self._end_cell()
            if self._row is None:
                self._row = []
            self._cell = []
        elif tag == 'br' and self._cell is not None:
            self._cell.append('\n')

    def handle_endtag(self, tag):
        if tag in ('td', 'th'):
            self._end_cell()
        elif tag in ('tr', 'table'):
            self._end_row()

    def handle_data(self, data):
        if self._cell is not None:
            self._cell.append(re.sub(r'\s+', ' ', data))    # only <br> breaks a line

    def _end_cell(self):
        if self._cell is not None:
            self._row.append(''.join(self._cell))
            self._cell = None

    def _end_row(self):
        self._end_cell()
        if self._row:
            self.rows.append(self._row)
        self._row = None


def _lines(cell):
    return [' '.join(line.split()) for line in cell.split('\n') if line.strip()]


def _columns(row):
    """Where each field sits if `row` is the header row, else None."""
    labels = [' '.join(cell).lower() for cell in row]

    def first(test):
        return next((i for i, label in enumerate(labels) if test(label)), None)
    col = {'stock': first(lambda s: 'stock' in s),
           'desc': first(lambda s: s == 'description'),
           'ex': first(lambda s: s.replace(' ', '-').startswith('ex-date')),
           'book': first(lambda s: 'book clos' in s)}
    return None if None in col.values() else col


def parse(page):
    """(meta, entries): one entry per row of HKEX's table, in HKEX's order."""
    cells = _Cells()
    cells.feed(page)
    cells.close()
    rows = [[_lines(c) for c in r] for r in cells.rows]
    for h, header in enumerate(rows):
        col = _columns(header)
        if col:
            break
    else:
        raise Stop('the HKEX page layout has changed - no header row with '
                   'Stock / Description / Ex-Date / Book Closing Date')

    entries, stock = [], ('', '')
    for row in rows[h + 1:]:
        if len(row) != len(header) or row == header:
            continue                                   # spacer rows
        if not any(line.strip('-') for cell in row for line in cell):
            continue                                   # the ------ underline
        if row[col['stock']]:                          # blank = same stock as the row above
            name = ' '.join(row[col['stock']])
            m = re.match(r'(.*?)\s*\((\d+)\)$', name)
            stock = (m.group(1), m.group(2)) if m else (name, '')
        entries.append({'flag': ['*'] in row,          # HKEX's own asterisk column
                        'name': stock[0], 'code': stock[1],
                        'desc': row[col['desc']],
                        'ex': ' '.join(row[col['ex']]),
                        'book': row[col['book']]})
    if not entries:
        raise Stop('the HKEX page has a header but no entries under it')

    date = re.search(r'(?m)^\s*Date\s*:\s*(\d{1,2}/\d{1,2}/\d{4})', page)
    flag_to = re.search(r'up to\s+(\d{1,2}/\d{1,2}/\d{4})', page)
    meta = {'date': dt.datetime.strptime(date.group(1), '%d/%m/%Y').date() if date else None,
            'flag_to': flag_to.group(1) if flag_to else '',
            'labels': {k: header[i] for k, i in col.items()}}
    return meta, entries


def keep(entries):
    """Every row of every stock that has at least one row mentioning a KEYWORD.  HKEX
    lists a stock with two entitlements on two rows (CITIC BANK: a rights issue and a
    dividend with an RMB option) - both rows stay, even when only one of them mentions it."""
    out = []
    for _, rows in itertools.groupby(entries, key=lambda e: (e['name'], e['code'])):
        rows = [dict(e, hits=mentions(e)) for e in rows]
        if any(e['hits'] for e in rows):
            out += rows
    return out


def mentions(entry):
    """The KEYWORDS its Description contains, any case.  Checked with the lines
    joined by a space and glued together, so a word HKEX wraps mid-way still counts."""
    spaced = ' '.join(entry['desc']).upper()
    glued = ''.join(entry['desc']).upper()
    return [k for k in KEYWORDS if k.upper() in spaced or k.upper() in glued]


# ----------------------------------------------------------------- email ---
def _br(lines):
    return '<br>'.join(html.escape(line) for line in lines)


def _day(d):
    return f'{DAYS[d.weekday()]} {d:%d/%m/%Y}'


def _listed(meta):
    return _day(meta['date']) if meta['date'] else 'undated'


def _tally(kept):
    return ' · '.join(f"{k.title()} {sum(k in e['hits'] for e in kept)}" for k in KEYWORDS)


def build_email(meta, kept, fetched):
    """(subject, html).  Outlook draws mail with Word, so: tables and inline styles only."""
    listed = _listed(meta)
    d = meta['date'] or fetched
    subject = SUBJECT.format(date=f'{d.day:02d} {MONTHS[d.month - 1]} {d.year}')
    words = ' or '.join(f'“{k.title()}”' for k in KEYWORDS)
    counts = _tally(kept)
    para = f'margin:0 0 12px 0;{FONT}font-size:10pt;color:{INK};'
    note = f'margin:10px 0 0 0;{FONT}font-size:8pt;color:{MUTED};'

    # one fixed-width frame, so the text wraps with the table in Outlook too
    out = ['<table cellpadding="0" cellspacing="0" border="0" width="752" style="width:752px;">'
           f'<tr><td style="{FONT}">']
    if GREETING:
        out.append(f'<p style="{para}">{html.escape(GREETING)}</p>')
    list_name = f'HKEX\'s <b>Dividends &amp; Other Entitlements</b> list dated <b>{listed}</b>'
    if not kept:
        out.append(f'<p style="{para}">No entry on {list_name} mentions {words}.</p>')
    else:
        stocks = len({(e['name'], e['code']) for e in kept})
        out.append(f'<p style="{para}">Below {"are" if stocks > 1 else "is"} the <b>{stocks}</b> '
                   f'stock{"s" if stocks > 1 else ""} with an entry mentioning {words} on {list_name} '
                   f'&ndash; {len(kept)} row{"s" if len(kept) > 1 else ""} ({counts}), every row of '
                   f'each stock included.</p>')

        th = (f'{FONT}font-size:9pt;font-weight:bold;color:#ffffff;background:{HEAD};'
              f'padding:7px 10px;text-align:left;vertical-align:bottom;')
        labels = meta['labels']
        cols = [(labels['stock'], 140), (labels['desc'], 300), (labels['ex'], 52), (labels['book'], 160)]
        out.append('<table cellpadding="0" cellspacing="0" border="0" width="752" '
                   'style="border-collapse:collapse;width:752px;">')
        out.append(f'<tr><td width="12" style="{th}padding:7px 0 7px 8px;">&nbsp;</td>'
                   + ''.join(f'<td width="{w}" style="{th}">{_br(label)}</td>' for label, w in cols)
                   + '</tr>')
        band, previous = True, None
        for e in kept:
            if (e['name'], e['code']) != previous:     # one shade per stock, as HKEX groups them
                band, previous = not band, (e['name'], e['code'])
            td = (f'{FONT}font-size:9.5pt;color:{INK};background:{BAND if band else "#ffffff"};'
                  f'padding:7px 10px;vertical-align:top;border-bottom:1px solid {RULE};'
                  f'line-height:14px;mso-line-height-rule:exactly;')
            flag = (f'<td style="{td}padding:7px 0 7px 8px;color:{FLAG};font-weight:bold;">*</td>'
                    if e['flag'] else f'<td style="{td}padding:7px 0 7px 8px;">&nbsp;</td>')
            stock = f'<b>{html.escape(e["name"])}</b>'
            if e['code']:
                stock += f'<br><span style="color:{MUTED};">({e["code"]})</span>'
            ex = html.escape(e['ex']) or f'<span style="color:{MUTED};">&mdash;</span>'
            dated = re.search(r'\d', ' '.join(e['book']))
            book = _br(e['book']) if dated else f'<span style="color:{MUTED};">{_br(e["book"])}</span>'
            out.append(f'<tr>{flag}<td style="{td}">{stock}</td><td style="{td}">{_br(e["desc"])}</td>'
                       f'<td style="{td}white-space:nowrap;">{ex}</td>'
                       f'<td style="{td}white-space:nowrap;">{book}</td></tr>')
        out.append('</table>')
        if meta['flag_to'] and any(e['flag'] for e in kept):
            out.append(f'<p style="{note}"><b style="color:{FLAG};">*</b> '
                       f'HKEX flag: effective book closing date commencing up to {meta["flag_to"]}.</p>')

    out.append('</td></tr></table>')
    return subject, ''.join(out)


def _page(subject, fragment):
    return (f'<html><head><meta charset="utf-8"><title>{html.escape(subject)}</title></head>'
            f'<body style="margin:0;padding:16px;background:#ffffff;">{fragment}</body></html>')


# ----------------------------------------------------------------- draft ---
def outlook_draft(subject, fragment):
    """Save the draft in Outlook and open it.  Raises if Outlook is not reachable."""
    import win32com.client
    mail = win32com.client.Dispatch('Outlook.Application').CreateItem(0)   # 0 = olMailItem
    mail.To = TO
    mail.CC = CC
    mail.Subject = subject
    try:
        mail.GetInspector                  # touching it makes Outlook add the default signature
        signature = mail.HTMLBody or ''
    except Exception:
        signature = ''
    body = re.search(r'<body[^>]*>', signature, flags=re.I)
    mail.HTMLBody = (signature[:body.end()] + fragment + signature[body.end():]
                     if body else _page(subject, fragment))
    mail.Save()
    if OPEN_DRAFT:
        mail.Display()
    return 'Outlook - saved to Drafts' + (' and opened' if OPEN_DRAFT else '')


def _write(folder, name, data):
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, name)
    with open(path, 'wb') as fh:
        fh.write(data)
    return path


def eml_draft(subject, fragment, stamp):
    """Write an .eml that opens as an unsent draft, and open it.  (path, opened)"""
    msg = EmailMessage()
    msg['Subject'] = subject
    if TO:
        msg['To'] = TO.replace(';', ',')
    if CC:
        msg['Cc'] = CC.replace(';', ',')
    msg['X-Unsent'] = '1'                  # Outlook opens it as an editable draft
    msg.set_content(_page(subject, fragment), subtype='html')
    name = f'hkex_rights_options_{stamp}.eml'
    try:
        path = _write(OUT_DIR, name, bytes(msg))
    except OSError:                        # e.g. a read-only share
        path = _write(tempfile.gettempdir(), name, bytes(msg))
    if not OPEN_DRAFT:
        return path, False
    try:
        if sys.platform == 'win32':
            os.startfile(path)
        else:
            subprocess.run(['open' if sys.platform == 'darwin' else 'xdg-open', path], check=True)
        return path, True
    except (OSError, subprocess.CalledProcessError):
        return path, False                 # no mail app for .eml - the file is still there


def make_draft(subject, fragment, stamp):
    try:
        return outlook_draft(subject, fragment)
    except ImportError:
        why = 'pywin32 not installed' if sys.platform == 'win32' else 'Outlook automation is Windows-only'
    except Exception as e:
        why = f'Outlook not reachable: {e}'
    path, opened = eml_draft(subject, fragment, stamp)
    return f'.eml {"written and opened" if opened else "written"} ({why}): {path}'


# ------------------------------------------------------------------- run ---
def _show(fragment):
    """Draw the email under the Jupyter cell.  False outside Jupyter."""
    try:
        from IPython import get_ipython
        from IPython.display import HTML, display
    except ImportError:
        return False
    if type(get_ipython()).__name__ != 'ZMQInteractiveShell':
        return False
    display(HTML(f'<div style="background:#ffffff;padding:14px 18px;border:1px solid {RULE};'
                 f'max-width:820px;">{fragment}</div>'))
    return True


def as_text(kept):
    """The kept rows as plain text, for a terminal."""
    rows = [(f"{'*' if e['flag'] else ' '} {e['name']} ({e['code']})", e['ex'] or '-',
             ' '.join(e['book']), ' '.join(e['desc'])) for e in kept]
    if not rows:
        return ''
    w = [max(len(r[i]) for r in rows) for i in range(3)]
    return '\n'.join(f'{a:<{w[0]}}  {b:<{w[1]}}  {c:<{w[2]}}  {d}' for a, b, c, d in rows)


def run():
    """Fetch the page, keep the RIGHT / OPTION entries, draft the email.  Returns the kept entries."""
    fetched = dt.datetime.now(HKT)
    try:
        page, updated, note = fetch()
        meta, entries = parse(page)
    except Stop as e:
        print(f'Stopped: {e}.  Nothing drafted.')
        return []
    kept = keep(entries)
    subject, fragment = build_email(meta, kept, fetched)
    draft = make_draft(subject, fragment, (meta['date'] or fetched).strftime('%Y%m%d'))

    counts = _tally(kept)
    facts = [('HKEX list', f'dated {_listed(meta)}'),
             ('Updated', f'{_day(updated)} {updated:%H:%M} HKT  (HKEX\'s own timestamp for the page)'
                         if updated else 'HKEX did not say'),
             ('Fetched', f'{_day(fetched)} {fetched:%H:%M} HKT  (fresh copy, no cache)'),
             ('Entries', f'{len(entries)} on the list'),
             ('Kept', f'{len({(e["name"], e["code"]) for e in kept})} stocks, {len(kept)} rows  ({counts})'),
             ('Subject', subject),
             ('Draft', draft)]
    if note:
        facts.append(('Note', note))
    for label, value in facts:
        print(f'{label:<10} {value}')
    if not _show(fragment):
        print('\n' + as_text(kept))
    return kept


if __name__ == '__main__':
    run()

## Tool 2 · Dividend currency elections, last five years
- Source: HKEX's own [filings search](https://www1.hkexnews.hk/search/titlesearch.xhtml?lang=en) - Circulars + Announcements and Notices. **Every company in the table comes from HKEX**; its date opens the form on hkexnews.hk.
- HKEX allows 12 months per all-stock search, so five one-year searches are merged - one row per company.
- Counted: forms titled as currency elections ("Dividend Currency Election Form", "RMB ... Dividend Election Form") and plain "Election Form"s only when the form offers two or more currencies (scrip forms drop out).
- Each company's latest form is read for the currencies it offers - not every currency election is HKD/RMB (HSBC: HKD/USD/GBP), so **HKD & RMB** companies are listed first.

In [ ]:
# TOOL 2 - HKEX dividend currency elections, last five years.  Edit CONFIG if you like, then Run this cell.
import collections
import concurrent.futures
import csv
import datetime as dt
import gzip
import html
import http.client
import importlib
import io
import json
import logging
import os
import re
import ssl
import subprocess
import sys
import tempfile
import time
import urllib.parse
import urllib.request
import warnings

# ================================================================ CONFIG ===
CATEGORIES = ('Circulars',                # HKEX headline categories, by HKEX's own names.  Most
              'Announcements and Notices')   # forms are circulars; CHINA VANKE, S HARBOURHOLD and
SEARCH_TITLES = ('Currency',                 # CHINA MOBILE filed some as announcements
                 'Election Form')         # searched in titles; every hit is then classified
YEARS = 5
READ_FORMS = True                         # read the forms for the currencies they offer
AUTO_INSTALL = True                       # pip-install pypdf the first time if it is missing
try:
    HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:                         # pasted straight into a notebook cell
    HERE = os.getcwd()
OUT_DIR = os.path.join(HERE, 'hkex_out')  # a CSV and a Markdown copy of the table, per run date

# ================================================================ ENGINE ===
SITE = 'https://www1.hkexnews.hk'
MAX_ROWS = 1000                           # HKEX returns at most 1,000 hits per search
HKT = dt.timezone(dt.timedelta(hours=8), 'HKT')
UA = ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
      '(KHTML, like Gecko) Chrome/124.0 Safari/537.36')
NO_CACHE = {'Cache-Control': 'no-cache', 'Pragma': 'no-cache'}
CURRENCIES = (('HKD', r'\bHKD\b|HK\$|HONG KONG DOLLAR|\bHK DOLLAR'),
              ('RMB', r'\bRMB\b|RENMINBI|\bCNY\b'),
              ('USD', r'\bUSD\b|US\$|UNITED STATES DOLLAR|\bU\.?S\.? DOLLAR'),
              ('GBP', r'\bGBP\b|STERLING|£'),
              ('EUR', r'\bEUR\b|\bEUROS?\b'),
              ('JPY', r'\bJPY\b|JAPANESE YEN'),
              ('SGD', r'\bSGD\b|SINGAPORE DOLLAR'),
              ('AUD', r'\bAUD\b|AUSTRALIAN DOLLAR'),
              ('CAD', r'\bCAD\b|CANADIAN DOLLAR'))
SKIP = re.compile(r'SCRIP|REINVEST|(?:SHARE|CASH) ALTERNATIVE|BONUS|NOTIFICATION LETTER|'
                  r'CORPORATE COMMUNICATION|MEANS OF RECEIPT|POLL RESULT|PROXY FORM')
ELECT = re.compile(r'(?<!RE-)\bELECT(?:ION)?S?\b')    # not "re-election" of directors
FONT = 'font-family:Arial,Helvetica,sans-serif;'
INK, MUTED, RULE, HEAD, BAND, YES, NO = '#1f2937', '#6b7280', '#dfe4ea', '#1f3a5f', '#f3f6fa', '#1e7b4f', '#b45309'


class Stop(Exception):
    """A failure one sentence explains - printed instead of a traceback."""


# ----------------------------------------------------------------- fetch ---
_route = {}                               # what worked first time is used from then on


def get(url):
    """The bytes at `url`: Python first; on Windows, if Python cannot get out (an office
    proxy that wants a sign-in, say), Windows' own web stack - the one the browser uses."""
    if _route.get('windows'):
        return _windows_get(url)
    try:
        return _python_get(url)
    except (OSError, http.client.HTTPException) as e:
        reason = getattr(e, 'reason', e)
        if sys.platform != 'win32':
            raise Stop(f'could not reach HKEX ({reason}) - check the network or proxy, then run again')
        try:
            data = _windows_get(url)
        except Exception as e2:
            raise Stop(f'could not reach HKEX (Python: {reason}; Windows: {e2}) - '
                       f'check the network, then run again')
        _route['windows'] = True
        return data


def _python_get(url):
    """Certificates: the system's, then certifi's, then none - these are public read-only
    pages, better fetched unchecked than not at all."""
    contexts = [_route['context']] if 'context' in _route else _contexts()
    request = urllib.request.Request(url, headers=dict(NO_CACHE, **{'User-Agent': UA}))
    for context in contexts:
        try:
            with urllib.request.urlopen(request, timeout=30, context=context) as response:
                data = response.read()
            break
        except OSError as e:
            if not isinstance(getattr(e, 'reason', e), ssl.SSLCertVerificationError):
                raise
            refused = e
    else:
        raise refused                     # every certificate route refused
    _route['context'] = context
    return gzip.decompress(data) if data[:2] == b'\x1f\x8b' else data


def _contexts():
    out = [ssl.create_default_context()]
    try:
        import certifi
        out.append(ssl.create_default_context(cafile=certifi.where()))
    except ImportError:
        pass
    return out + [ssl._create_unverified_context()]


def _windows_get(url):
    """Through MSXML/WinINet: the browser's proxy settings and sign-in."""
    import win32com.client
    xhr = win32com.client.Dispatch('MSXML2.XMLHTTP.6.0')
    xhr.open('GET', url, False)
    for name, value in NO_CACHE.items():
        xhr.setRequestHeader(name, value)
    xhr.send()
    if xhr.status != 200:
        raise OSError(f'HTTP {xhr.status}')
    return bytes(xhr.responseBody)


def _json(url):
    return json.loads(get(url).decode('utf-8-sig'))


# ---------------------------------------------------------------- search ---
def category_codes(names=CATEGORIES):
    """HKEX's code for each headline category name, from HKEX's own list."""
    known = {c['name'].lower(): c['code'] for c in _json(SITE + '/ncms/script/eds/tierone_e.json')}
    missing = [n for n in names if n.lower() not in known]
    if missing:
        raise Stop(f'HKEX has no headline category called {missing} - check CATEGORIES')
    return [(n, known[n.lower()]) for n in names]


def windows(today, years=YEARS):
    """`years` back-to-back windows of 12 months, newest first, the first ending today."""
    out, end = [], today
    for _ in range(years):
        try:
            start = end.replace(year=end.year - 1) + dt.timedelta(days=1)
        except ValueError:                # 29 February
            start = end.replace(year=end.year - 1, day=28) + dt.timedelta(days=1)
        out.append((start, end))
        end = start - dt.timedelta(days=1)
    return out


def _query(t1code, start, end, title, rows):
    q = dict(sortDir=0, sortByOptions='DateTime', category=0, market='SEHK', stockId=-1,
             documentType=-1, fromDate=f'{start:%Y%m%d}', toDate=f'{end:%Y%m%d}', title=title,
             searchType=1, t1code=t1code, t2Gcode=-2, t2code=-2, rowRange=rows, lang='E')
    reply = _json(SITE + '/search/titleSearchServlet.do?' + urllib.parse.urlencode(q))
    result = reply.get('result')
    hits = json.loads(result) if isinstance(result, str) and result not in ('', 'null') else []
    return hits, int(reply.get('recordCnt') or 0)


def search(t1code, start, end, title):
    """Every hit for `title` in one category between two dates, inclusive.  HKEX answers a
    window it will not serve with nothing at all, and caps a hit list at 1,000 - either
    way the window is split in half and searched again."""
    hits, count = _query(t1code, start, end, title, MAX_ROWS)
    refused = count == 0 and _query(t1code, start, end, '', 1)[1] == 0   # nothing at all = refused
    if (refused or count > len(hits)) and (end - start).days > 31:
        middle = start + (end - start) // 2
        return search(t1code, start, middle, title) + search(t1code, middle + dt.timedelta(days=1), end, title)
    if count > len(hits):
        raise Stop(f'more than {MAX_ROWS} hits for "{title}" in {start:%d/%m/%Y}-{end:%d/%m/%Y}')
    return hits


def kind(title):
    """'says' - the title says it is a currency election;  'check' - an election form whose
    text must show a choice of currencies;  None - anything else."""
    t = title.upper()
    if SKIP.search(t) or not ELECT.search(t):
        return None
    if 'CURRENCY' in t or (re.search(r'\bRMB\b|RENMINBI', t) and 'ELECTION FORM' in t):
        return 'says'
    return 'check' if 'ELECTION FORM' in t else None


def _split(field):
    return [html.unescape(' '.join(p.split())) for p in re.split(r'(?i)<br\s*/?>', field or '')]


def document(hit, category):
    codes, names = _split(hit.get('STOCK_CODE')), _split(hit.get('STOCK_NAME'))
    names += [''] * (len(codes) - len(names))
    link = hit.get('FILE_LINK') or ''
    title = html.unescape(' '.join((hit.get('TITLE') or '').split()))
    return {'id': hit.get('NEWS_ID') or link,
            'codes': codes, 'names': names,
            'when': dt.datetime.strptime(hit['DATE_TIME'], '%d/%m/%Y %H:%M'),
            'title': title, 'kind': kind(title),
            'link': link if link.startswith('http') else SITE + link,
            'category': category}


def collect(cats, spans):
    """Every filing of the window whose title marks it 'says' or 'check' - one search per
    title x category x 12-month window, four at a time."""
    tasks = [(title, name, code, start, end) for title in SEARCH_TITLES for name, code in cats
             for start, end in spans]
    docs = {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
        for batch in pool.map(lambda t: [document(h, t[1]) for h in search(t[2], t[3], t[4], t[0])], tasks):
            for d in batch:
                if d['kind'] and d['id'] not in docs:
                    docs[d['id']] = d
    return docs


# ----------------------------------------------------------------- forms ---
def pdf_reader_installed():
    for name in ('pypdf', 'PyPDF2'):
        try:
            __import__(name)
            return True
        except ImportError:
            pass
    return False


def ensure_reader():
    """(True, note) once a PDF reader imports - installing pypdf first when AUTO_INSTALL."""
    if pdf_reader_installed():
        return True, ''
    fix = 'run  pip install pypdf  once, then Run again'
    if not AUTO_INSTALL:
        return False, f'pypdf is not installed - {fix}'
    print('Installing pypdf (reads the forms) - first run only ...')
    try:
        done = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'pypdf'],
                              capture_output=True, text=True, timeout=300)
    except (OSError, subprocess.SubprocessError) as e:
        return False, f'could not install pypdf ({e}) - {fix}'
    importlib.invalidate_caches()
    if done.returncode == 0 and pdf_reader_installed():
        return True, 'pypdf installed'
    last = (done.stderr or done.stdout or 'pip failed').strip().splitlines()[-1]
    last = last.replace(sys.executable, 'python')          # "python: No module named pip"
    return False, f'could not install pypdf ({last[:160]}) - {fix}'


def _pdf_text(data, pages=8):
    try:
        from pypdf import PdfReader
    except ImportError:
        from PyPDF2 import PdfReader      # the same library under its older name
    for name in ('pypdf', 'PyPDF2'):
        logging.getLogger(name).setLevel(logging.ERROR)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return ' '.join((p.extract_text() or '') for p in PdfReader(io.BytesIO(data)).pages[:pages])


def read_currencies(doc):
    """(currencies the form lets holders choose, e.g. ['HKD', 'RMB'], or None;  why not read)."""
    try:
        data = get(doc['link'])
        if doc['link'].lower().endswith('.pdf'):
            text = _pdf_text(data)
        else:
            text = re.sub(r'<[^>]+>', ' ', data.decode('utf-8', 'replace'))
    except Exception as e:                # one unreadable form must not stop the list
        return None, f'could not read the form ({e})'
    text = ' '.join(html.unescape(text).split()).upper()
    if not text:
        return None, 'no text in the form (a scan?)'
    return offered(text), ''


def offered(text):
    """The currencies a form lets you choose.  The currency a dividend is DECLARED in is
    named too - but as an amount (RMB0.21), an exchange rate (HK$1.0 : RMB0.87) or
    "declared in ..." - so those mentions are dropped first.  E.g. HX BLDG MAT declares
    in RMB and offers HKD or USD: RMB must not count."""
    any_currency = '(?:' + '|'.join(pattern for _, pattern in CURRENCIES) + ')S?'
    text = re.sub(f'EXCHANGE RATE (?:OF |BETWEEN |FROM )?{any_currency}\\s*[\\d.,]*\\s*'
                  f'(?:TO|:|=|AND|AGAINST|INTO)\\s*{any_currency}', ' ', text)      # "HK$1.0 : RMB0.87"
    text = re.sub(f'(?:DECLARED|DENOMINATED) IN {any_currency}', ' ', text)
    return [k for k, pattern in CURRENCIES if re.search(f'(?:{pattern})(?!\\s*[\\d.,]*\\d)', text)]


def _read_all(docs):
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
        for d, (cur, why) in zip(docs, pool.map(read_currencies, docs)):
            d['currencies'], d['why'] = cur, why


# --------------------------------------------------------------- company ---
def companies(docs):
    """One entry per company.  Codes filed together on one document are one company (its
    shares, an RMB counter, its notes), and a code keeps its row through a change of name.
    Only the FORMS are counted when a company files them: NIRAKU files "How to elect ... the
    currency" next to every "Currency Election Form", SY HOLDINGS announcements next to its
    forms - counting both would count every dividend twice."""
    parent = {}

    def root(c):
        parent.setdefault(c, c)
        while parent[c] != c:
            parent[c] = parent[parent[c]]
            c = parent[c]
        return c
    for d in docs:
        for c in d['codes'][1:]:
            parent[root(c)] = root(d['codes'][0])
    groups = collections.defaultdict(list)
    for d in docs:
        groups[root(d['codes'][0])].append(d)

    out = []
    for filed in groups.values():
        filed.sort(key=lambda d: d['when'], reverse=True)                  # newest first
        group = [d for d in filed if 'FORM' in d['title'].upper()] or filed
        code = collections.Counter(d['codes'][0] for d in group).most_common(1)[0][0]
        names = [d['names'][d['codes'].index(code)] for d in group if code in d['codes']]
        out.append({'code': code, 'name': names[0],
                    'formerly': [n for i, n in enumerate(names) if n and n != names[0] and n not in names[:i]],
                    'docs': group, 'filed': filed, 'latest': group[0],
                    'years': collections.Counter(d['when'].year for d in group),
                    'currencies': None, 'why': 'not read'})
    return out


# ----------------------------------------------------------------- table ---
def _both(c):
    return None if c['currencies'] is None else ('HKD' in c['currencies'] and 'RMB' in c['currencies'])


def _order(rows):
    rank = {True: 0, False: 1, None: 2}
    return sorted(rows, key=lambda c: (rank[_both(c)], c['code']))


LABELS = {True: 'HKD and RMB on the latest form', False: 'Other currencies (no RMB on the latest form)',
          None: 'Latest form not read'}


def as_html(rows, span, years, counts):
    th = (f'{FONT}font-size:8.5pt;font-weight:bold;color:#ffffff;background:{HEAD};'
          f'padding:6px 8px;text-align:left;vertical-align:bottom;white-space:nowrap;')
    heads = (['Stock Code', 'Stock Name', 'HKD &amp; RMB', 'Currencies<br>(latest form)']
             + [f'{y}' for y in years] + ['Total', 'Latest form'])
    out = [f'<div style="{FONT}color:{INK};">',
           '<div style="font-size:12pt;font-weight:bold;margin:0 0 4px 0;">'
           'HKEX dividend currency elections &ndash; one row per company</div>',
           f'<div style="font-size:9pt;color:{MUTED};margin:0 0 10px 0;">{html.escape(counts)}</div>',
           '<table cellpadding="0" cellspacing="0" border="0" style="border-collapse:collapse;">',
           '<tr>' + ''.join(f'<td style="{th}{"text-align:center;" if h[:2].isdigit() or h == "Total" else ""}">'
                            f'{h}</td>' for h in heads) + '</tr>']
    section, band = object(), False
    for c in _order(rows):
        if _both(c) != section:
            section = _both(c)
            n = sum(1 for r in rows if _both(r) == section)
            out.append(f'<tr><td colspan="{len(heads)}" style="{FONT}font-size:9pt;font-weight:bold;'
                       f'color:{INK};padding:12px 8px 4px 8px;border-bottom:2px solid {HEAD};">'
                       f'{LABELS[section]} ({n})</td></tr>')
        band = not band
        td = (f'{FONT}font-size:9pt;color:{INK};background:{BAND if band else "#ffffff"};padding:5px 8px;'
              f'border-bottom:1px solid {RULE};vertical-align:top;')
        name = f'<b>{html.escape(c["name"])}</b>'
        if c['formerly']:
            name += f'<br><span style="color:{MUTED};">formerly {html.escape(", ".join(c["formerly"]))}</span>'
        mark = {True: f'<b style="color:{YES};">Yes</b>', False: f'<b style="color:{NO};">No</b>',
                None: f'<span style="color:{MUTED};" title="{html.escape(c["why"])}">?</span>'}[_both(c)]
        cur = ' · '.join(c['currencies']) if c['currencies'] else f'<span style="color:{MUTED};">&ndash;</span>'
        cells = [f'<td style="{td}">{c["code"]}</td>', f'<td style="{td}">{name}</td>',
                 f'<td style="{td}text-align:center;">{mark}</td>', f'<td style="{td}white-space:nowrap;">{cur}</td>']
        for y in years:
            k = c['years'].get(y, 0)
            cells.append(f'<td style="{td}text-align:center;{"font-weight:bold;" if k else ""}">{k or ""}</td>')
        latest = c['latest']
        cells.append(f'<td style="{td}text-align:center;">{len(c["docs"])}</td>')
        cells.append(f'<td style="{td}white-space:nowrap;"><a href="{html.escape(latest["link"])}" '
                     f'title="{html.escape(latest["title"])}">{latest["when"]:%d/%m/%Y}</a></td>')
        out.append('<tr>' + ''.join(cells) + '</tr>')
    out.append('</table>')
    out.append(f'<div style="font-size:8.5pt;color:{MUTED};margin-top:8px;">Years count the forms filed in '
               f'that calendar year within {html.escape(span)}. Currencies are the ones the latest form lets '
               f'holders choose; the date opens it.</div></div>')
    return ''.join(out)


def _write_csv(rows, years, path):
    header = (['stock_code', 'bbg_ticker', 'stock_name', 'formerly', 'hkd_and_rmb', 'currencies']
              + [str(y) for y in years] + ['total', 'latest_date', 'latest_title', 'latest_link'])
    with open(path, 'w', newline='', encoding='utf-8-sig') as fh:        # -sig: Excel reads it as UTF-8
        w = csv.writer(fh)
        w.writerow(header)
        for c in _order(rows):
            w.writerow([c['code'], f'{int(c["code"])} HK' if c['code'].isdigit() else '', c['name'],
                        '; '.join(c['formerly']), {True: 'Yes', False: 'No', None: '?'}[_both(c)],
                        ' '.join(c['currencies'] or [])]
                       + [c['years'].get(y, 0) for y in years]
                       + [len(c['docs']), f'{c["latest"]["when"]:%d/%m/%Y}', c['latest']['title'],
                          c['latest']['link']])


def _write_md(rows, years, span, counts, path):
    head = ['Code', 'Company', 'HKD & RMB', 'Currencies'] + [str(y) for y in years] + ['Total', 'Latest form']
    lines = [f'# HKEX dividend currency elections, {span}', '', counts, '',
             'Years count the forms filed in that calendar year. Currencies are the ones the latest '
             'form lets holders choose; the date links to it.']
    section = object()
    for c in _order(rows):
        if _both(c) != section:
            section = _both(c)
            n = sum(1 for r in rows if _both(r) == section)
            lines += ['', f'## {LABELS[section]} ({n})', '', '| ' + ' | '.join(head) + ' |',
                      '|' + '---|' * len(head)]
        name = c['name'] + (f' (formerly {", ".join(c["formerly"])})' if c['formerly'] else '')
        cells = ([c['code'], name, {True: 'Yes', False: 'No', None: '?'}[_both(c)],
                  ' · '.join(c['currencies'] or []) or '–']
                 + [str(c['years'].get(y) or '') for y in years]
                 + [str(len(c['docs'])), f"[{c['latest']['when']:%d/%m/%Y}]({c['latest']['link']})"])
        lines.append('| ' + ' | '.join(x.replace('|', '/') for x in cells) + ' |')
    with open(path, 'w', encoding='utf-8') as fh:
        fh.write('\n'.join(lines) + '\n')


def save(rows, years, span, counts, stamp):
    """CSV + Markdown copies of the table in OUT_DIR - or in TEMP if that cannot be written."""
    for folder in (OUT_DIR, tempfile.gettempdir()):
        try:
            os.makedirs(folder, exist_ok=True)
            base = os.path.join(folder, f'hkex_currency_election_{stamp}')
            _write_csv(rows, years, base + '.csv')
            _write_md(rows, years, span, counts, base + '.md')
            return base + '.csv  (+ .md)'
        except OSError as e:
            error = e
    return f'not saved ({error})'


# ------------------------------------------------------------------- run ---
def _show(fragment):
    try:
        from IPython import get_ipython
        from IPython.display import HTML, display
    except ImportError:
        return False
    if type(get_ipython()).__name__ != 'ZMQInteractiveShell':
        return False
    display(HTML(f'<div style="background:#ffffff;padding:14px 18px;">{fragment}</div>'))
    return True


def run():
    """Search, classify, read the forms, draw the table, save it.  Returns one dict per company."""
    started, now = time.time(), dt.datetime.now(HKT)
    spans = windows(now.date())
    first, last = spans[-1][0], spans[0][1]
    span = f'{first:%d/%m/%Y} - {last:%d/%m/%Y}'
    try:
        cats = category_codes()
        print(f'Searching HKEX: {len(SEARCH_TITLES) * len(cats) * len(spans)} one-year searches ...')
        docs = collect(cats, spans)
    except Stop as e:
        print(f'Stopped: {e}.  Nothing listed.')
        return []

    readable, note = ensure_reader() if READ_FORMS else (False, 'forms not read (READ_FORMS = False)')
    checks = [d for d in docs.values() if d['kind'] == 'check']
    if readable:
        print(f'Reading {len(checks)} plain election forms for a choice of currencies ...')
        _read_all(checks)
    kept = [d for d in docs.values() if d['kind'] == 'says' or len(d.get('currencies') or []) >= 2]
    rows = companies(kept)
    if readable:
        todo = [c['latest'] for c in rows if 'currencies' not in c['latest']]
        print(f'Reading the latest form of {len(todo)} companies ...')
        _read_all(todo)
        for c in rows:
            cur, why = c['latest']['currencies'], c['latest']['why']
            if cur is not None and len(cur) < 2:            # a currency election names two or more
                cur, why = None, 'the form text names no choice of currencies'
            c['currencies'], c['why'] = cur, why
    else:
        for c in rows:
            c['why'] = note

    years = [y for y in range(first.year, last.year + 1) if any(c['years'].get(y) for c in rows)]
    tally = collections.Counter(_both(c) for c in rows)
    says = sum(1 for d in kept if d['kind'] == 'says')
    counts = f'{len(rows)} companies · {len(kept)} forms · {span} · run {now:%d/%m/%Y %H:%M} HKT'
    yours = sum(1 for c in rows if any(d['category'] == 'Circulars' and 'CURRENCY ELECTION' in d['title'].upper()
                                       for d in c['filed']))
    facts = [('Source', 'HKEX filings search - www1.hkexnews.hk/search/titlesearch.xhtml (every row links to its form)'),
             ('Search', 'titles with "' + '" or "'.join(SEARCH_TITLES) + '" in ' + ' + '.join(n for n, _ in cats)),
             ('Window', f'{span}  ({len(spans)} x 12-month all-stock searches per title and category)'),
             ('Forms', f'{len(kept)} currency elections  ({says} titled as such, {len(kept) - says} plain '
                       f'election forms offering 2+ currencies, {len(checks) - (len(kept) - says)} other election forms dropped)'
                       if readable else f'{says} titled as currency elections  ({len(checks)} plain election forms NOT checked)'),
             ('Companies', f'{len(rows)}  (one row each: stock codes and renames merged)'),
             ('Your search', f'{yours} of the {len(rows)} have a Circular titled "Currency Election"; the other '
                             f'{len(rows) - yours} filed the form as an announcement or under another title'),
             ('HKD & RMB', f'{tally[True]} yes · {tally[False]} other currencies · {tally[None]} not read')]
    if note:
        facts.append(('Note', note))
    facts.append(('Saved', save(rows, years, span, counts, f'{now:%Y-%m-%d}')))
    facts.append(('Took', f'{time.time() - started:.0f} s'))
    for label, value in facts:
        print(f'{label:<13}{value}')
    if not _show(as_html(rows, span, years, counts)):
        for c in _order(rows):
            both = {True: 'Yes', False: 'No ', None: ' ? '}[_both(c)]
            print(f"{c['code']}  {c['name']:<18} {both}  {' '.join(c['currencies'] or []):<14}"
                  + ' '.join(f"{c['years'].get(y, 0) or '.':>2}" for y in years)
                  + f"  latest {c['latest']['when']:%d/%m/%Y}")
    return rows


if __name__ == '__main__':
    run()